# Optional Colab fallback: verified PlantVillage training

Use this only if the local CPU training run cannot finish. This notebook runs the repository's grouped, duplicate-checked trainer; it does not claim that a Colab run has occurred. Before starting, upload the current AgriVision project folder (including `ml/disease/prepare_plantvillage_subset.py` and `ml/disease/train_verified.py`) to `MyDrive/AgriVision`. The notebook downloads the public 868 MB CC0 no-augmentation PlantVillage archive and writes model/report artifacts back to that folder.

A PlantVillage test score is not evidence of field-photo performance.

In [ ]:
%pip install -q tensorflow-cpu==2.20.0 scikit-learn==1.9.1 numpy==2.5.3 pandas==3.0.6 scipy==1.18.1 Pillow==12.3.0 matplotlib==3.11.2

In [ ]:
from google.colab import drive
from pathlib import Path
import requests
import subprocess
import sys

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/AgriVision')
prepare_script = PROJECT_ROOT / 'ml' / 'disease' / 'prepare_plantvillage_subset.py'
train_script = PROJECT_ROOT / 'ml' / 'disease' / 'train_verified.py'
if not prepare_script.is_file() or not train_script.is_file():
    raise FileNotFoundError('Upload the current AgriVision project folder to MyDrive/AgriVision first.')

archive_dir = PROJECT_ROOT / 'data' / 'raw' / 'source_archives'
archive_dir.mkdir(parents=True, exist_ok=True)
archive = archive_dir / 'plant_village_without_augmentation.zip'
dataset_url = 'https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded'
if not archive.is_file():
    with requests.get(dataset_url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with archive.open('xb') as output:
            for block in response.iter_content(chunk_size=1024 * 1024):
                if block:
                    output.write(block)

dataset_dir = Path('/content/PlantVillage')
if not dataset_dir.exists() or not any(dataset_dir.iterdir()):
    subprocess.run([
        sys.executable, str(prepare_script), '--archive', str(archive),
        '--destination', str(dataset_dir)
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Reusing existing extracted PlantVillage subset in /content/PlantVillage')

subprocess.run([
    sys.executable, str(train_script), '--data-dir', str(dataset_dir),
    '--model-dir', str(PROJECT_ROOT / 'models' / 'disease'),
    '--report-dir', str(PROJECT_ROOT / 'reports' / 'disease'),
    '--max-per-class', '250', '--epochs', '12'
], cwd=PROJECT_ROOT, check=True)